# Ciencia de Datos con Python - Semana 3
**Cuaderno de Trabajo:** `Laboratorio_Semana_03_Perfilamiento_Calidad_Datos.ipynb`  
**Tema:** Perfilamiento de datos, detección de anomalías y evaluación de calidad.

---

## Propósito
En este laboratorio realizaremos la inspección profunda y el perfilamiento de calidad de datos sobre un dataset con errores deliberados (nulos, duplicados, inconsistencias y valores fuera de rango).


## 1. Carga de Datos con Rutas Relativas
Importamos las librerías necesarias y cargamos el dataset con errores desde `data/raw/`.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

# Configuración de rutas relativas
BASE_DIR = Path("..")
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

# Ruta del archivo con errores
ruta_dataset = DATA_RAW / "seguimiento_academico_semana3_con_errores.csv"

# Carga del dataset
df = pd.read_csv(ruta_dataset)
print("✅ Dataset cargado exitosamente. Dimensiones iniciales:", df.shape)
df.head()

✅ Dataset cargado exitosamente. Dimensiones iniciales: (20, 9)


,Documento,Nombre,Programa,Ficha,Nota Python,Nota Datos,Asistencia (%),Ciudad,Fecha Actualizacion
0,1001,Ana,ADSO,287001,4.2,4.0,92.0,Medellín,2026-10-12
1,1002,Carlos,ADSO,287001,3.8,3.6,85.0,Bello,2026-10-11
2,1003,María,ADSO,287001,4.7,4.8,96.0,Itagüí,2026-10-12
3,1004,Luis,ADSO,287001,2.9,3.1,70.0,Medellín,2026-10-10
4,1005,Sofía,ADSO,287001,5.0,4.9,98.0,Envigado,2026-10-09


## 2. Diagnóstico de Calidad de Datos

Analizaremos las dimensiones, tipos de datos, cantidad de valores faltantes (nulos) y presencia de registros duplicados.

In [6]:
print("--- Resumen Estadístico de Variables Numéricas ---")
display(df.describe())

# Detección de notas fuera del rango válido (0.0 a 5.0)
print("\n--- Registros con notas fuera de rango (< 0.0 o > 5.0) ---")
notas_invalidas = df[(df["Nota Python"] < 0) | (df["Nota Python"] > 5) | (df["Nota Datos"] < 0) | (df["Nota Datos"] > 5)]
display(notas_invalidas)

# Detección de asistencias fuera de rango (< 0% o > 100%)
print("\n--- Registros con asistencia fuera de rango (< 0 o > 100) ---")
asistencia_invalida = df[(df["Asistencia (%)"] < 0) | (df["Asistencia (%)"] > 100)]
display(asistencia_invalida)

--- Resumen Estadístico de Variables Numéricas ---


,Documento,Ficha,Nota Datos,Asistencia (%)
count,20.000000,20.0,19.000000,19.000000
mean,1009.000000,287001.0,3.952632,82.631579
std,5.341003,0.0,0.586296,22.415794
min,1001.000000,287001.0,3.000000,-4.000000
25%,1004.750000,287001.0,3.600000,81.500000
50%,1008.500000,287001.0,4.000000,88.000000
75%,1013.250000,287001.0,4.300000,92.500000
max,1018.000000,287001.0,4.900000,102.000000



--- Registros con notas fuera de rango (< 0.0 o > 5.0) ---


TypeError: '<' not supported between instances of 'str' and 'int'

## 3. Detección de Inconsistencias y Valores Fuera de Rango

Evaluaremos valores atípicos o erróneos en columnas numéricas y formatos incoherentes en texto.

In [10]:
# 1. Eliminación de filas completamente duplicadas
df_limpio = df.drop_duplicates().copy()

# 2. Convertir columnas numéricas a float (forzando errores a NaN)
columnas_numericas = ["Nota Python", "Nota Datos", "Asistencia (%)"]
for col in columnas_numericas:
    # Si las notas tienen comas como separador decimal, las reemplazamos por puntos
    if df_limpio[col].dtype == "object":
        df_limpio[col] = df_limpio[col].astype(str).str.replace(",", ".")
    df_limpio[col] = pd.to_numeric(df_limpio[col], errors="coerce")

# 3. Corrección/Filtrado de valores fuera de rango (0.0 a 5.0 en notas, 0 a 100 en asistencia)
df_limpio = df_limpio[
    (df_limpio["Nota Python"].isna()) | ((df_limpio["Nota Python"] >= 0) & (df_limpio["Nota Python"] <= 5))
]
df_limpio = df_limpio[
    (df_limpio["Nota Datos"].isna()) | ((df_limpio["Nota Datos"] >= 0) & (df_limpio["Nota Datos"] <= 5))
]
df_limpio = df_limpio[
    (df_limpio["Asistencia (%)"].isna()) | ((df_limpio["Asistencia (%)"] >= 0) & (df_limpio["Asistencia (%)"] <= 100))
]

# 4. Imputación de valores faltantes (NaN) con el promedio de la columna
df_limpio["Nota Python"] = df_limpio["Nota Python"].fillna(df_limpio["Nota Python"].mean())
df_limpio["Nota Datos"] = df_limpio["Nota Datos"].fillna(df_limpio["Nota Datos"].mean())
df_limpio["Asistencia (%)"] = df_limpio["Asistencia (%)"].fillna(df_limpio["Asistencia (%)"].mean())

print("✅ Dataset limpiado exitosamente. Nuevas dimensiones:", df_limpio.shape)
df_limpio.head()

✅ Dataset limpiado exitosamente. Nuevas dimensiones: (17, 9)


,Documento,Nombre,Programa,Ficha,Nota Python,Nota Datos,Asistencia (%),Ciudad,Fecha Actualizacion
0,1001,Ana,ADSO,287001,4.2,4.0,92.0,Medellín,2026-10-12
1,1002,Carlos,ADSO,287001,3.8,3.6,85.0,Bello,2026-10-11
2,1003,María,ADSO,287001,4.7,4.8,96.0,Itagüí,2026-10-12
3,1004,Luis,ADSO,287001,2.9,3.1,70.0,Medellín,2026-10-10
4,1005,Sofía,ADSO,287001,5.0,4.9,98.0,Envigado,2026-10-09


In [5]:
print("Columnas reales del CSV:")
print(df.columns.tolist())

Columnas reales del CSV:
['Documento', 'Nombre', 'Programa', 'Ficha', 'Nota Python', 'Nota Datos', 'Asistencia (%)', 'Ciudad', 'Fecha Actualizacion']


## 5. Exportación de Resultados

Guardamos el dataset procesado y limpio en `data/processed/seguimiento_semana3_limpio.csv`.

In [11]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ruta_salida = DATA_PROCESSED / "seguimiento_semana3_limpio.csv"

df_limpio.to_csv(ruta_salida, index=False, encoding="utf-8")
print("✅ Dataset limpio exportado exitosamente en:", ruta_salida)

✅ Dataset limpio exportado exitosamente en: ..\data\processed\seguimiento_semana3_limpio.csv


## 6. Conclusiones del Perfilamiento de Calidad

1. **Principales problemas detectados:** Se encontraron registros duplicados, valores nulos en calificaciones/asistencia y notas/asistencias fuera de rango.
2. **Acciones correctivas:** Se estandarizaron los tipos de datos a numéricos, se eliminaron filas duplicadas y valores atípicos imposibles, e imputamos los valores faltantes mediante el promedio de cada columna.
3. **Impacto:** El proceso garantiza que las métricas e informes de seguimiento académico reflejen la realidad de los aprendices sin distorsiones.
